In [9]:
import pandas as pd

# Load all 3 cleaned files
ch = pd.read_csv("ch_cleaned.csv")
hh = pd.read_csv("hh_cleaned.csv")
hl = pd.read_csv("hl_cleaned.csv")

print(f"ch shape: {ch.shape}")
print(f"hh shape: {hh.shape}")
print(f"hl shape: {hl.shape}")

ch shape: (67563, 18)
hh shape: (65660, 10)
hl shape: (479999, 11)


In [10]:
# Checking columns in each
print("\nch columns:", ch.columns.tolist())
print("\nhh columns:", hh.columns.tolist())
print("\nhl columns:", hl.columns.tolist())


ch columns: ['cluster_id', 'household_id', 'child_age_months', 'child_sex', 'breastfeeding', 'haz_score', 'waz_score', 'whz_score', 'stunted', 'underweight', 'wasted', 'mother_education', 'urban_rural', 'district', 'wealth_index', 'disability', 'province', 'birth_weight_known']

hh columns: ['cluster_id', 'household_id', 'housing_quality', 'water_source', 'sanitation_type', 'education_level', 'urban_rural', 'district', 'wealth_index', 'province']

hl columns: ['cluster_id', 'household_id', 'child_sex', 'child_age_months', 'mother_education', 'household_education', 'urban_rural', 'district', 'wealth_index', 'disability', 'province']


In [11]:

# Drop duplicate columns from hh and hl
# before merging (already exist in ch)

hh_clean = hh.drop(columns=['urban_rural', 'district', 'wealth_index', 'province'])
hl_clean = hl.drop(columns=['child_sex', 'child_age_months', 'urban_rural',
                              'district', 'wealth_index', 'province', 'disability'])

print("hh columns after drop:", hh_clean.columns.tolist())
print("hl columns after drop:", hl_clean.columns.tolist())


# Merge ch + hh first

merged = ch.merge(hh_clean, on=['cluster_id', 'household_id'], how='left')
print(f"\nAfter ch + hh merge: {merged.shape}")


# Merge with hl
merged = merged.merge(hl_clean, on=['cluster_id', 'household_id'], how='left')
print(f"After ch + hh + hl merge: {merged.shape}")


# Check result

print(f"\nFinal columns: {merged.columns.tolist()}")
print(f"\nMissing values:")
print(merged.isnull().sum())
print(f"\nSample:")
print(merged.head())

hh columns after drop: ['cluster_id', 'household_id', 'housing_quality', 'water_source', 'sanitation_type', 'education_level']
hl columns after drop: ['cluster_id', 'household_id', 'mother_education', 'household_education']

After ch + hh merge: (194536, 22)
After ch + hh + hl merge: (4624673, 24)

Final columns: ['cluster_id', 'household_id', 'child_age_months', 'child_sex', 'breastfeeding', 'haz_score', 'waz_score', 'whz_score', 'stunted', 'underweight', 'wasted', 'mother_education_x', 'urban_rural', 'district', 'wealth_index', 'disability', 'province', 'birth_weight_known', 'housing_quality', 'water_source', 'sanitation_type', 'education_level', 'mother_education_y', 'household_education']

Missing values:
cluster_id             0
household_id           0
child_age_months       0
child_sex              0
breastfeeding          0
haz_score              0
waz_score              0
whz_score              0
stunted                0
underweight            0
wasted                 0
mother

In [12]:

# FIX 1 — Fix duplicate rows from hl merge
# hl has multiple members per household
# we only want one row per child


# Check how many duplicate rows we have
print(f"Rows before dedup: {merged.shape[0]:,}")
merged = merged.drop_duplicates(subset=['cluster_id', 'household_id',
                                         'child_age_months', 'child_sex'])
print(f"Rows after dedup: {merged.shape[0]:,}")


# FIX 2 — Fix duplicate mother_education
# Keep mother_education_x (from ch file)
# Drop mother_education_y (from hl file)
merged = merged.drop(columns=['mother_education_y'])
merged = merged.rename(columns={'mother_education_x': 'mother_education'})


# Final check

print(f"\nFinal shape: {merged.shape}")
print(f"Columns: {merged.columns.tolist()}")
print(f"\nMissing values:")
print(merged.isnull().sum())


Rows before dedup: 4,624,673
Rows after dedup: 66,393

Final shape: (66393, 23)
Columns: ['cluster_id', 'household_id', 'child_age_months', 'child_sex', 'breastfeeding', 'haz_score', 'waz_score', 'whz_score', 'stunted', 'underweight', 'wasted', 'mother_education', 'urban_rural', 'district', 'wealth_index', 'disability', 'province', 'birth_weight_known', 'housing_quality', 'water_source', 'sanitation_type', 'education_level', 'household_education']

Missing values:
cluster_id             0
household_id           0
child_age_months       0
child_sex              0
breastfeeding          0
haz_score              0
waz_score              0
whz_score              0
stunted                0
underweight            0
wasted                 0
mother_education       0
urban_rural            0
district               0
wealth_index           0
disability             0
province               0
birth_weight_known     0
housing_quality        0
water_source           0
sanitation_type        0
educat

In [13]:
print(f"Stunting rate: {merged['stunted'].mean()*100:.1f}%")
print(f"\nStunting distribution:")
print(merged['stunted'].value_counts())

print(f"\nProvince distribution:")
print(merged['province'].value_counts())

Stunting rate: 42.5%

Stunting distribution:
stunted
0    38208
1    28185
Name: count, dtype: int64

Province distribution:
province
0    25205
1    23654
2    17534
Name: count, dtype: int64


In [15]:
merged.to_csv("final_dataset.csv", index=False)
print(f"Saved! Shape: {merged.shape}")
print(merged.dtypes)

Saved! Shape: (66393, 23)
cluster_id             float64
household_id           float64
child_age_months       float64
child_sex                int64
breastfeeding          float64
haz_score              float64
waz_score              float64
whz_score              float64
stunted                  int64
underweight              int64
wasted                   int64
mother_education       float64
urban_rural              int64
district               float64
wealth_index           float64
disability             float64
province                 int64
birth_weight_known       int64
housing_quality        float64
water_source           float64
sanitation_type        float64
education_level        float64
household_education    float64
dtype: object
